# 04 — Modelagem

## Modelagem, otimização de hiperparâmetros e interpretação

### Planejamento

1. **Preparar os dados:** carregar as bases de treino, teste OOT e submissão, respeitando as partições já definidas. Não imputar valores ausentes.
2. **Treinar o XGBoost baseline:** usar seed fixa e parâmetros iniciais; registrar AUC no treino e no teste OOT.
3. **Treinar o LightGBM baseline:** usar a mesma seed e avaliar com as mesmas métricas.
4. **Comparar os baselines:** analisar desempenho e diferença entre AUC de treino e OOT, considerando que o OOT já foi consultado nas análises anteriores.
5. **Otimizar o melhor modelo com Optuna:** usar validação temporal interna walk-forward para orientar a busca de hiperparâmetros, sem selecionar parâmetros pela avaliação OOT.
6. **Avaliar o modelo otimizado:** medir o desempenho final no teste OOT e comparar com o baseline correspondente.
7. **Avaliar o modelo otimizado no conjunto de contratos rejeitados:** comparar o score com treino/submissao. Esperado que a média de scores seja maior.
7. **Interpretar o modelo com SHAP:** examinar a importância global das variáveis e a direção de seus efeitos, com foco no modelo selecionado.

### Princípios

- Manter a separação temporal existente entre treino e teste OOT.
- Preservar valores ausentes; não aplicar imputação.
- Usar categóricas nativas dos modelos quando suportadas.
- Fixar e registrar as seeds para reprodutibilidade.
- Documentar parâmetros, métricas e decisões de modelagem.

### Para lembrar:

- Avaliar usar QuantileTransformer do sklearn
- **Avaliar o modelo otimizado no conjunto de contratos rejeitados:** pode aplicar o modelo em abt_completa filtrada para o conjunto de rejeitados.
- Mostrar a evolução do AUC por safra! Pode mostrar no treino mesmo, dentro do cross validation.
- No mercado, usa-se o quantiletransforme por safra, assim vc ve o perfil da probabilidade agrupado para ver se é estável ao longo do tempo!

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score

# ============================================================
# 1. Carregamento das bases de modelagem
# ============================================================

CAMINHO_PROCESSADOS = Path("../dados/processados")

CAMINHO_TREINO_TESTE = (
    CAMINHO_PROCESSADOS / "abt_modelagem_treino_teste.parquet"
)
CAMINHO_SUBMISSAO = (
    CAMINHO_PROCESSADOS / "abt_modelagem_submissao.parquet"
)

COL_TARGET = "OVER60MOB12"
COL_PARTICAO = "particao"
COL_ID_CONTRATO = "id_contrato"
COL_ID_CLIENTE = "id_cliente"

FEATURES_MODELO = [
    "tipo_contrato",
    "qtd_filhos",
    "qtd_membros_familia",
    "renda_anual",
    "tipo_renda",
    "ocupacao",
    "tipo_organizacao",
    "nivel_educacao",
    "estado_civil",
    "tipo_moradia",
    "possui_carro",
    "possui_imovel",
    "nota_regiao_cliente",
    "nota_regiao_cliente_cidade",
    "parc_atrasadas_por_contrato_exec",
    "pct_parc_atrasadas_360d",
    "atraso_medio_720d",
    "qtd_contratos_executados",
    "pct_parc_atrasadas_180d",
    "pct_parc_pagas_a_menos_720d",
    "parc_gt90_por_contrato_exec",
    "razao_credito_atual_media_720d",
    "parc_gt30_por_mes_rel",
    "qtd_parc_vencidas_360d",
    "pct_parc_atraso_gt30_180d",
    "qtd_aprovados_360d",
    "idade_anos",
]

# Variáveis reservadas para análises e construção de políticas.
# Não serão usadas como preditoras nos modelos.
VARIAVEIS_POLITICA = [
    "valor_credito",
    "valor_bem",
    "valor_parcela",
    "razao_credito_renda",
    "razao_parcela_renda",
    "razao_credito_bem",
]

FEATURES_CATEGORICAS = [
    "tipo_contrato",
    "tipo_renda",
    "ocupacao",
    "tipo_organizacao",
    "nivel_educacao",
    "estado_civil",
    "tipo_moradia",
    "possui_carro",
    "possui_imovel",
]

SEED = 42

assert len(FEATURES_MODELO) == 27
assert len(set(FEATURES_MODELO)) == len(FEATURES_MODELO)
assert len(VARIAVEIS_POLITICA) == 6
assert len(set(VARIAVEIS_POLITICA)) == len(VARIAVEIS_POLITICA)
assert not set(FEATURES_MODELO).intersection(VARIAVEIS_POLITICA)
assert set(FEATURES_CATEGORICAS).issubset(FEATURES_MODELO)

abt_modelagem_treino_teste = pd.read_parquet(CAMINHO_TREINO_TESTE)
abt_modelagem_submissao = pd.read_parquet(CAMINHO_SUBMISSAO)


# ============================================================
# 2. Validações estruturais
# ============================================================

COLUNAS_OBRIGATORIAS_TREINO = [
    COL_ID_CONTRATO,
    COL_ID_CLIENTE,
    COL_TARGET,
    COL_PARTICAO,
    *FEATURES_MODELO,
    *VARIAVEIS_POLITICA,
]

COLUNAS_OBRIGATORIAS_SUBMISSAO = [
    COL_ID_CLIENTE,
    *FEATURES_MODELO,
    *VARIAVEIS_POLITICA,
]

COLUNAS_AUSENTES_TREINO = [
    coluna
    for coluna in COLUNAS_OBRIGATORIAS_TREINO
    if coluna not in abt_modelagem_treino_teste.columns
]

COLUNAS_AUSENTES_SUBMISSAO = [
    coluna
    for coluna in COLUNAS_OBRIGATORIAS_SUBMISSAO
    if coluna not in abt_modelagem_submissao.columns
]

if COLUNAS_AUSENTES_TREINO:
    raise KeyError(
        "Colunas ausentes na base treino/teste: "
        f"{COLUNAS_AUSENTES_TREINO}"
    )

if COLUNAS_AUSENTES_SUBMISSAO:
    raise KeyError(
        "Colunas ausentes na base de submissão: "
        f"{COLUNAS_AUSENTES_SUBMISSAO}"
    )

if COL_TARGET in abt_modelagem_submissao.columns:
    raise ValueError("A base de submissão não deve conter o target.")

if abt_modelagem_treino_teste[COL_TARGET].isna().any():
    raise ValueError("Há valores nulos no target da base treino/teste.")

if abt_modelagem_treino_teste[COL_ID_CONTRATO].isna().any():
    raise ValueError("Há id_contrato nulo na base treino/teste.")

if abt_modelagem_treino_teste[COL_ID_CONTRATO].duplicated().any():
    raise ValueError("Há contratos duplicados na base treino/teste.")

if abt_modelagem_treino_teste[COL_PARTICAO].isna().any():
    raise ValueError("Há valores nulos na coluna de partição.")

PARTICOES_ESPERADAS = {"treino", "teste_oot"}
PARTICOES_ENCONTRADAS = set(
    abt_modelagem_treino_teste[COL_PARTICAO].unique()
)

if PARTICOES_ENCONTRADAS != PARTICOES_ESPERADAS:
    raise ValueError(
        "Partições inesperadas. "
        f"Esperadas: {PARTICOES_ESPERADAS}; "
        f"encontradas: {PARTICOES_ENCONTRADAS}."
    )


# ============================================================
# 3. Separação das partições já definidas
# ============================================================

base_treino = abt_modelagem_treino_teste.loc[
    abt_modelagem_treino_teste[COL_PARTICAO].eq("treino")
].copy()

base_teste_oot = abt_modelagem_treino_teste.loc[
    abt_modelagem_treino_teste[COL_PARTICAO].eq("teste_oot")
].copy()

if len(base_treino) != 83_335:
    raise ValueError(
        "Esperadas 83.335 linhas de treino; "
        f"encontradas {len(base_treino):,}."
    )

if len(base_teste_oot) != 20_822:
    raise ValueError(
        "Esperadas 20.822 linhas de teste OOT; "
        f"encontradas {len(base_teste_oot):,}."
    )

# As matrizes X contêm exclusivamente as 27 features do modelo.
X_treino = base_treino.loc[:, FEATURES_MODELO].copy()
y_treino = base_treino[COL_TARGET].astype("int8")

X_teste_oot = base_teste_oot.loc[:, FEATURES_MODELO].copy()
y_teste_oot = base_teste_oot[COL_TARGET].astype("int8")

X_submissao = abt_modelagem_submissao.loc[:, FEATURES_MODELO].copy()

# Mantém as variáveis de política separadas das matrizes de modelagem.
politicas_treino = base_treino.loc[:, VARIAVEIS_POLITICA].copy()
politicas_teste_oot = base_teste_oot.loc[:, VARIAVEIS_POLITICA].copy()
politicas_submissao = abt_modelagem_submissao.loc[
    :, VARIAVEIS_POLITICA
].copy()

# Confirma que nenhuma variável reservada para política entrou em X.
for nome, X in [
    ("treino", X_treino),
    ("teste OOT", X_teste_oot),
    ("submissão", X_submissao),
]:
    variaveis_indesejadas = [
        coluna
        for coluna in VARIAVEIS_POLITICA
        if coluna in X.columns
    ]
    if variaveis_indesejadas:
        raise ValueError(
            f"Variáveis de política incluídas indevidamente em X de {nome}: "
            f"{variaveis_indesejadas}"
        )


# ============================================================
# 4. Alinhamento dos tipos categóricos
# ============================================================

# Padroniza o domínio categórico entre treino, OOT e submissão.
# Isso inclui categorias observadas somente fora do treino.
for coluna in FEATURES_CATEGORICAS:
    categorias = pd.concat(
        [
            X_treino[coluna].astype("string"),
            X_teste_oot[coluna].astype("string"),
            X_submissao[coluna].astype("string"),
        ],
        ignore_index=True,
    ).dropna().unique()

    dtype_categoria = pd.CategoricalDtype(
        categories=sorted(categorias.tolist()),
        ordered=False,
    )

    X_treino[coluna] = (
        X_treino[coluna].astype("string").astype(dtype_categoria)
    )
    X_teste_oot[coluna] = (
        X_teste_oot[coluna].astype("string").astype(dtype_categoria)
    )
    X_submissao[coluna] = (
        X_submissao[coluna].astype("string").astype(dtype_categoria)
    )


# ============================================================
# 5. Resumo para conferência
# ============================================================

print(f"Features do modelo: {len(FEATURES_MODELO)}")
print(f"Variáveis de política: {len(VARIAVEIS_POLITICA)}")
print(
    f"Treino: {X_treino.shape} | "
    f"taxa de maus: {y_treino.mean():.4%}"
)
print(
    f"Teste OOT: {X_teste_oot.shape} | "
    f"taxa de maus: {y_teste_oot.mean():.4%}"
)
print(f"Submissão: {X_submissao.shape}")
print(
    "Valores ausentes nas features de treino:",
    f"{X_treino.isna().sum().sum():,}",
)
print("\nTipos das features de treino:")
print(X_treino.dtypes.value_counts())

print("\nVariáveis reservadas para políticas:")
print(VARIAVEIS_POLITICA)

Features do modelo: 27
Variáveis de política: 6
Treino: (83335, 27) | taxa de maus: 6.9695%
Teste OOT: (20822, 27) | taxa de maus: 7.8667%
Submissão: (50400, 27)
Valores ausentes nas features de treino: 442,730

Tipos das features de treino:
float32     12
Int64        2
float64      2
category     2
int64        2
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64

Variáveis reservadas para políticas:
['valor_credito', 'valor_bem', 'valor_parcela', 'razao_credito_renda', 'razao_parcela_renda', 'razao_credito_bem']


In [2]:
# ============================================================
# Diagnóstico de ausências e categorias fora do treino
# Não altera as bases.
# ============================================================

features_categoricas = X_treino.select_dtypes(
    include=["category", "object", "string"]
).columns.tolist()

features_numericas = [
    coluna
    for coluna in FEATURES_MODELO
    if coluna not in features_categoricas
]

print(f"Features categóricas ({len(features_categoricas)}):")
print(features_categoricas)

print(f"\nFeatures numéricas ({len(features_numericas)}):")
print(features_numericas)

# Ausências por feature e partição.
resumo_ausencias = pd.DataFrame({
    "treino": X_treino.isna().mean(),
    "teste_oot": X_teste_oot.isna().mean(),
    "submissao": X_submissao.isna().mean(),
}).sort_values("treino", ascending=False)

print("\nPercentual de valores ausentes:")
display(resumo_ausencias)

# Categorias observadas na submissão que não existem no treino.
for coluna in features_categoricas:
    categorias_treino = set(X_treino[coluna].dropna().astype("string").unique())
    categorias_submissao = set(X_submissao[coluna].dropna().astype("string").unique())
    categorias_novas = sorted(categorias_submissao - categorias_treino)

    print(
        f"{coluna}: {len(categorias_treino)} categorias no treino, "
        f"{len(categorias_novas)} categorias novas na submissão"
    )

    if categorias_novas:
        print(f"  Novas categorias: {categorias_novas[:20]}")

Features categóricas (9):
['tipo_contrato', 'tipo_renda', 'ocupacao', 'tipo_organizacao', 'nivel_educacao', 'estado_civil', 'tipo_moradia', 'possui_carro', 'possui_imovel']

Features numéricas (18):
['qtd_filhos', 'qtd_membros_familia', 'renda_anual', 'nota_regiao_cliente', 'nota_regiao_cliente_cidade', 'parc_atrasadas_por_contrato_exec', 'pct_parc_atrasadas_360d', 'atraso_medio_720d', 'qtd_contratos_executados', 'pct_parc_atrasadas_180d', 'pct_parc_pagas_a_menos_720d', 'parc_gt90_por_contrato_exec', 'razao_credito_atual_media_720d', 'parc_gt30_por_mes_rel', 'qtd_parc_vencidas_360d', 'pct_parc_atraso_gt30_180d', 'qtd_aprovados_360d', 'idade_anos']

Percentual de valores ausentes:


,treino,teste_oot,submissao
pct_parc_atrasadas_180d,0.622356,0.518298,0.456587
pct_parc_atraso_gt30_180d,0.622356,0.518298,0.456587
razao_credito_atual_media_720d,0.601260,0.471232,0.369425
pct_parc_atrasadas_360d,0.568009,0.456680,0.350238
atraso_medio_720d,0.509270,0.388291,0.231290
pct_parc_pagas_a_menos_720d,0.509270,0.388291,0.231290
parc_gt90_por_contrato_exec,0.455451,0.269667,0.076865
parc_atrasadas_por_contrato_exec,0.455451,0.269667,0.076865
parc_gt30_por_mes_rel,0.423256,0.247815,0.042202
ocupacao,0.323682,0.313370,0.310575


tipo_contrato: 3 categorias no treino, 0 categorias novas na submissão
tipo_renda: 8 categorias no treino, 1 categorias novas na submissão
  Novas categorias: ['empresario']
ocupacao: 18 categorias no treino, 0 categorias novas na submissão
tipo_organizacao: 57 categorias no treino, 0 categorias novas na submissão
nivel_educacao: 5 categorias no treino, 0 categorias novas na submissão
estado_civil: 5 categorias no treino, 0 categorias novas na submissão
tipo_moradia: 6 categorias no treino, 0 categorias novas na submissão
possui_carro: 2 categorias no treino, 0 categorias novas na submissão
possui_imovel: 2 categorias no treino, 0 categorias novas na submissão


In [3]:
# ============================================================
# Padroniza categorias entre treino, teste OOT e submissão
# Inclui categorias que aparecem apenas no teste/submissão,
# como "empresario", sem imputar valores ausentes.
# ============================================================
FEATURES_CATEGORICAS = [
    "tipo_contrato",
    "tipo_renda",
    "ocupacao",
    "tipo_organizacao",
    "nivel_educacao",
    "estado_civil",
    "tipo_moradia",
    "possui_carro",
    "possui_imovel",
]

FEATURES_NUMERICAS = [
    coluna
    for coluna in FEATURES_MODELO
    if coluna not in FEATURES_CATEGORICAS
]

bases_modelagem = {
    "treino": X_treino,
    "teste_oot": X_teste_oot,
    "submissao": X_submissao,
}

for coluna in FEATURES_CATEGORICAS:
    # Junta as categorias observadas nas três bases, ignorando nulos.
    categorias_unicas = pd.concat(
        [
            base[coluna].astype("string")
            for base in bases_modelagem.values()
        ],
        ignore_index=True,
    ).dropna().unique()

    dtype_categoria = pd.CategoricalDtype(
        categories=sorted(categorias_unicas.tolist()),
        ordered=False,
    )

    # Converte primeiro para string para evitar o aviso de categorias novas.
    for base in bases_modelagem.values():
        base[coluna] = base[coluna].astype("string").astype(dtype_categoria)

# Confere categorias e tipos após a padronização.
print("Tipos categóricos após padronização:")
print(X_treino[FEATURES_CATEGORICAS].dtypes)

print("\nCategorias presentes apenas fora do treino:")
for coluna in FEATURES_CATEGORICAS:
    categorias_treino = set(X_treino[coluna].dropna().astype("string").unique())
    categorias_oot = set(X_teste_oot[coluna].dropna().astype("string").unique())
    categorias_submissao = set(
        X_submissao[coluna].dropna().astype("string").unique()
    )

    novas_oot = sorted(categorias_oot - categorias_treino)
    novas_submissao = sorted(categorias_submissao - categorias_treino)

    if novas_oot or novas_submissao:
        print(
            f"{coluna}: "
            f"novas no OOT={novas_oot}; "
            f"novas na submissão={novas_submissao}"
        )

# Verifica infinitos em variáveis numéricas.
for nome, base in bases_modelagem.items():
    valores_infinitos = base[FEATURES_NUMERICAS].isin(
        [float("inf"), float("-inf")]
    ).sum()

    colunas_com_inf = valores_infinitos[valores_infinitos.gt(0)]
    if not colunas_com_inf.empty:
        print(f"\nValores infinitos em {nome}:")
        print(colunas_com_inf)

print("\nPercentual de ausentes nas features numéricas — treino:")
print((X_treino[FEATURES_NUMERICAS].isna().mean() * 100).sort_values(ascending=False))

Tipos categóricos após padronização:
tipo_contrato       category
tipo_renda          category
ocupacao            category
tipo_organizacao    category
nivel_educacao      category
estado_civil        category
tipo_moradia        category
possui_carro        category
possui_imovel       category
dtype: object

Categorias presentes apenas fora do treino:
tipo_renda: novas no OOT=[]; novas na submissão=['empresario']

Percentual de ausentes nas features numéricas — treino:
pct_parc_atraso_gt30_180d           62.235555
pct_parc_atrasadas_180d             62.235555
razao_credito_atual_media_720d      60.125997
pct_parc_atrasadas_360d             56.800864
atraso_medio_720d                   50.926981
pct_parc_pagas_a_menos_720d         50.926981
parc_gt90_por_contrato_exec         45.545089
parc_atrasadas_por_contrato_exec    45.545089
parc_gt30_por_mes_rel               42.325553
idade_anos                           0.926381
renda_anual                          0.199196
qtd_filhos       

In [4]:
# ============================================================
# Split já definido no projeto: sem recriar as partições
# ============================================================

SEED = 42

COL_TARGET = "OVER60MOB12"
COL_PARTICAO = "particao"

FEATURES_CATEGORICAS = [
    "tipo_contrato",
    "tipo_renda",
    "ocupacao",
    "tipo_organizacao",
    "nivel_educacao",
    "estado_civil",
    "tipo_moradia",
    "possui_carro",
    "possui_imovel",
]

FEATURES_MODELO = [
    "tipo_contrato",
    "qtd_filhos",
    "qtd_membros_familia",
    "renda_anual",
    "tipo_renda",
    "ocupacao",
    "tipo_organizacao",
    "nivel_educacao",
    "estado_civil",
    "tipo_moradia",
    "possui_carro",
    "possui_imovel",
    "nota_regiao_cliente",
    "nota_regiao_cliente_cidade",
    "parc_atrasadas_por_contrato_exec",
    "pct_parc_atrasadas_360d",
    "atraso_medio_720d",
    "qtd_contratos_executados",
    "pct_parc_atrasadas_180d",
    "pct_parc_pagas_a_menos_720d",
    "parc_gt90_por_contrato_exec",
    "razao_credito_atual_media_720d",
    "parc_gt30_por_mes_rel",
    "qtd_parc_vencidas_360d",
    "pct_parc_atraso_gt30_180d",
    "qtd_aprovados_360d",
    "idade_anos",
]

# Confirma estrutura e partições esperadas.
assert len(FEATURES_MODELO) == 27
assert len(set(FEATURES_MODELO)) == 27
assert set(FEATURES_CATEGORICAS).issubset(FEATURES_MODELO)
assert not abt_modelagem_treino_teste[COL_TARGET].isna().any()

base_treino = abt_modelagem_treino_teste.loc[
    abt_modelagem_treino_teste[COL_PARTICAO].eq("treino")
].copy()

base_teste_oot = abt_modelagem_treino_teste.loc[
    abt_modelagem_treino_teste[COL_PARTICAO].eq("teste_oot")
].copy()

X_treino = base_treino.loc[:, FEATURES_MODELO].copy()
y_treino = base_treino[COL_TARGET].astype("int8")

X_teste_oot = base_teste_oot.loc[:, FEATURES_MODELO].copy()
y_teste_oot = base_teste_oot[COL_TARGET].astype("int8")

X_submissao = abt_modelagem_submissao.loc[:, FEATURES_MODELO].copy()

# Uniformiza os domínios categóricos nos três conjuntos.
# Isso preserva categorias de submissão não observadas no treino,
# sem imputar valores nem transformar categorias em números.
for coluna in FEATURES_CATEGORICAS:
    categorias = pd.concat(
        [
            X_treino[coluna].astype("string"),
            X_teste_oot[coluna].astype("string"),
            X_submissao[coluna].astype("string"),
        ],
        ignore_index=True,
    ).dropna().unique()

    dtype_categoria = pd.CategoricalDtype(
        categories=sorted(categorias.tolist()),
        ordered=False,
    )

    X_treino[coluna] = X_treino[coluna].astype("string").astype(dtype_categoria)
    X_teste_oot[coluna] = (
        X_teste_oot[coluna].astype("string").astype(dtype_categoria)
    )
    X_submissao[coluna] = (
        X_submissao[coluna].astype("string").astype(dtype_categoria)
    )

assert len(X_treino) == 83_335
assert len(X_teste_oot) == 20_822
assert len(X_submissao) == 50_400

print(f"Treino: {X_treino.shape} | taxa de maus: {y_treino.mean():.4%}")
print(
    f"Teste OOT: {X_teste_oot.shape} | "
    f"taxa de maus: {y_teste_oot.mean():.4%}"
)
print(f"Submissão: {X_submissao.shape}")
print(f"Valores ausentes no treino: {X_treino.isna().sum().sum():,}")

Treino: (83335, 27) | taxa de maus: 6.9695%
Teste OOT: (20822, 27) | taxa de maus: 7.8667%
Submissão: (50400, 27)
Valores ausentes no treino: 442,730


In [5]:
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier


# ============================================================
# XGBoost: baseline com seed fixa
# ============================================================

modelo_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=10,
    subsample=0.90,
    colsample_bytree=0.90,
    reg_alpha=0.0,
    reg_lambda=1.0,
    tree_method="hist",
    enable_categorical=True,
    random_state=SEED,
    n_jobs=-1,
)

modelo_xgb.fit(X_treino, y_treino)

prob_xgb_treino = modelo_xgb.predict_proba(X_treino)[:, 1]
prob_xgb_oot = modelo_xgb.predict_proba(X_teste_oot)[:, 1]

pred_xgb_treino = (prob_xgb_treino >= 0.5).astype("int8")
pred_xgb_oot = (prob_xgb_oot >= 0.5).astype("int8")

print(f"XGBoost AUC treino:     {roc_auc_score(y_treino, prob_xgb_treino):.4f}")
print(f"XGBoost F1 treino:      {f1_score(y_treino, pred_xgb_treino, zero_division=0):.4f}")
print(f"XGBoost AUC teste OOT:  {roc_auc_score(y_teste_oot, prob_xgb_oot):.4f}")
print(f"XGBoost F1 teste OOT:   {f1_score(y_teste_oot, pred_xgb_oot, zero_division=0):.4f}")

XGBoost AUC treino:     0.7734
XGBoost F1 treino:      0.0021
XGBoost AUC teste OOT:  0.6627
XGBoost F1 teste OOT:   0.0000


In [6]:
# ============================================================
# LightGBM: baseline com seed fixa
# ============================================================

modelo_lgbm = LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=100,
    colsample_bytree=0.90,
    subsample=0.90,
    subsample_freq=1,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

modelo_lgbm.fit(
    X_treino,
    y_treino,
    categorical_feature=FEATURES_CATEGORICAS,
)

prob_lgbm_treino = modelo_lgbm.predict_proba(X_treino)[:, 1]
prob_lgbm_oot = modelo_lgbm.predict_proba(X_teste_oot)[:, 1]

pred_lgbm_treino = (prob_lgbm_treino >= 0.5).astype("int8")
pred_lgbm_oot = (prob_lgbm_oot >= 0.5).astype("int8")

print(f"LightGBM AUC treino:     {roc_auc_score(y_treino, prob_lgbm_treino):.4f}")
print(f"LightGBM F1 treino:      {f1_score(y_treino, pred_lgbm_treino, zero_division=0):.4f}")
print(f"LightGBM AUC teste OOT:  {roc_auc_score(y_teste_oot, prob_lgbm_oot):.4f}")
print(f"LightGBM F1 teste OOT:   {f1_score(y_teste_oot, pred_lgbm_oot, zero_division=0):.4f}")

LightGBM AUC treino:     0.8195
LightGBM F1 treino:      0.0058
LightGBM AUC teste OOT:  0.6662
LightGBM F1 teste OOT:   0.0000


## Comparação com o baseline V2

A comparação principal considera o cenário oficial, com embargo, e o teste OOT completo, composto por 20.822 contratos.

| Modelo / cenário | Recorte | n | AUC |
|---|---|---:|---:|
| Baseline V2 — com embargo (oficial) | Teste completo | 20.822 | 0,556 |
| Baseline V2 — com embargo | Novos | 8.765 | 0,541 |
| Baseline V2 — com embargo | Recorrentes | 12.057 | 0,574 |
| Baseline V2 — sem embargo (sensibilidade) | Teste completo | 20.822 | 0,568 |
| XGBoost baseline — com embargo | Teste completo | 20.822 | 0,6627 |
| LightGBM baseline — com embargo | Teste completo | 20.822 | 0,6662 |

### Análise

No teste OOT completo, o XGBoost obteve AUC de 0,6627, aumento absoluto de 0,1067 em relação ao baseline V2 oficial (0,556). O LightGBM obteve AUC de 0,6662, aumento absoluto de 0,1102. Os resultados indicam melhora na capacidade de ordenação dos contratos pelos modelos baseline atuais.

O baseline V2 sem embargo, com AUC de 0,568, é apresentado como análise de sensibilidade. Não substitui a referência oficial com embargo e deve ser interpretado como um cenário distinto.

O F1-score foi registrado apenas como métrica complementar nos baselines atuais, com limiar de 0,5. Não será utilizado nas próximas etapas de seleção ou otimização. O objetivo é desenvolver políticas de crédito que considerem scores e variáveis disponíveis, em vez de definir um único limiar de classificação.

Como o teste OOT já foi consultado em análises anteriores, os resultados devem ser interpretados como uma comparação retrospectiva. A otimização de hiperparâmetros será orientada por validação temporal interna.